# Example 11.5.1 / 1: A Logarithmic Allocation Model and Its Solver Route

Companion notebook for **11.5.1 Exponential Cones** in the AMPL lab Documentation, section "A Logarithmic Allocation Model and Its Solver Route". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Logarithmic Allocation Model and Its Solver Route

The diminishing-returns allocation from Nonlinear Models (AMPL lab documentation, section 11.2) bounds each project's utility
by a logarithm, $\mathit{Utility}_p\le\log(1+\mathit{Spend}_p)$. The same hypograph is written here
in the equivalent exponential form $1+\mathit{Spend}_p\ge\exp(\mathit{Utility}_p)$, the epigraph
pattern `affine >= exp(variable)` that the MP documentation shows being recognized as an
exponential cone. Ipopt solves it as an ordinary smooth NLP:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROJECTS;                  # candidate projects
param benefit {PROJECTS} > 0;  # benefit weight per project
param budget > 0;              # total spending available

var Spend {PROJECTS} >= 0;  # spending per project
var Utility {PROJECTS};     # log utility, bounded by ExpCone

maximize TotalUtility:  # weighted utility
    sum {p in PROJECTS} benefit[p] * Utility[p];

subject to Budget:  # total spending within budget
    sum {p in PROJECTS} Spend[p] <= budget;

subject to ExpCone {p in PROJECTS}:  # (Utility, 1, 1 + Spend) in the exponential cone
    1 + Spend[p] >= exp(Utility[p]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROJECTS"] = ["A", "B"]
ampl.param["benefit"] = {"A": 2, "B": 1}
ampl.param["budget"] = 5

In [5]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(ampl.var["Spend"].to_dict())
print(ampl.var["Utility"].to_dict())

solved
{'A': 3.6666666931104452, 'B': 1.3333333510261167}
{'A': 1.5404450475044915, 'B': 0.847297869748941}


### Pure AMPL: data.dat and model.run

Pure AMPL; `solve > ipopt.log;` sends Ipopt's licence banner and log to the file `ipopt.log`:

In [6]:
%%writefile data.dat
set PROJECTS := A B;
param benefit := A 2 B 1;
param budget := 5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
solve > ipopt.log;
display Spend, Utility;

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
:    Spend    Utility     :=
A   3.66667   1.54045
B   1.33333   0.847298
;



### Another way in amplpy

Recomputing the original logarithms from the returned spending, with the same `model.mod` and
`data.py`, confirms that each cone constraint is tight at the optimum,
$\log(4.666667)=1.540445$ and $\log(2.333333)=0.847298$:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROJECTS"] = ["A", "B"]
ampl.param["benefit"] = {"A": 2, "B": 1}
ampl.param["budget"] = 5

In [11]:
# run.py
import math

ampl.solve(solver="ipopt", verbose=False)
spend = ampl.var["Spend"].to_dict()
utility = ampl.var["Utility"].to_dict()
for p in spend:
    print(p, round(math.log(1 + spend[p]), 6), round(utility[p], 6))

A 1.540445 1.540445
B 0.847298 0.847298


### Another way in amplpy

Recognition is conditional. A solver may accept a native exponential cone, a driver may reformulate a supported expression, or the model may require a general nonlinear solver. The presence of `exp` or `log` in AMPL source alone does not prove the solver received a conic model; the solver's log does.

With the same `model.mod` and `data.py`, Gurobi's log, captured with `ampl.get_output("solve;")`,
reports the two `ExpCone` rows as general nonlinear constraints, not as cones. The MP option
`cvt:expcones=0`, which the MP documentation describes for solvers with a native exponential-cone
API such as MOSEK, changes nothing here because Gurobi has no such API:

In [12]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [13]:
# data.py
ampl.set["PROJECTS"] = ["A", "B"]
ampl.param["benefit"] = {"A": 2, "B": 1}
ampl.param["budget"] = 5

In [14]:
# run.py
ampl.option["solver"] = "gurobi"
for opts in ("outlev=1", "outlev=1 cvt:expcones=0"):
    ampl.option["gurobi_options"] = opts
    log = ampl.get_output("solve;")
    print(opts)
    for line in log.splitlines():
        if "has" in line and "constraint" in line and "objective" not in line:
            print("  ", line.strip())
    print("   TotalUtility =", round(ampl.obj["TotalUtility"].value(), 6))

outlev=1
   Model has 2 general nonlinear constraints (2 nonlinear terms)
   Presolved model has 2 nonlinear constraint(s)
   TotalUtility = 3.928188
outlev=1 cvt:expcones=0
   Model has 2 general nonlinear constraints (2 nonlinear terms)
   Presolved model has 2 nonlinear constraint(s)
   TotalUtility = 3.928188
